# 1 · Probability for robotics

**Recap** ([theory](https://github.com/Rudip1/learn-state-estimation/blob/main/1_theory/01_probability.md)).
A belief about a state is a probability density; a Gaussian $\mathcal N(\mu,\Sigma)$ is described by its mean
and covariance. A linear map $y = Ax+b$ sends it exactly to $\mathcal N(A\mu+b, A\Sigma A^T)$; a nonlinear map is
linearised at the mean, $\Sigma_y \approx J\Sigma J^T$, which can be badly wrong. New information enters through
Bayes' rule, $p(x\mid z) = \eta\, p(z\mid x)p(x)$; for two Gaussians this is the fusion
$K = \Sigma_a(\Sigma_a+\Sigma_b)^{-1}$, $\mu = \mu_a + K(\mu_b-\mu_a)$. The squared Mahalanobis distance
$d_M^2 = (x-\mu)^T\Sigma^{-1}(x-\mu)$ is $\chi^2_n$-distributed and is the basis of every gate and consistency test.

In [ ]:
# Installs the module when it is not available yet (e.g. on Colab). Building the C++ takes a few minutes.
try:
    import state_estimation
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-state-estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import state_estimation as se
from state_estimation import plotting

plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6, 4.5)})

## Sampling a Gaussian and its confidence ellipses

`se.sample_gaussian` draws $x = \mu + Lz$ with $\Sigma = LL^T$ (eq. 1.16). The $k$-sigma ellipse $d_M^2 \le k^2$
holds probability `chi2_cdf(k**2, 2)` — not the 68/95/99.7 % familiar from one dimension.

In [ ]:
rng = se.Rng(0)
mu = np.array([1.0, 0.5])
Sigma = np.array([[1.0, 0.6], [0.6, 0.5]])
X = se.sample_gaussian(mu, Sigma, 3000, rng)

ax = plotting.new_axes()
ax.plot(X[:, 0], X[:, 1], ".", ms=2, color="0.6")
for k in (1, 2, 3):
    p = se.chi2_cdf(k**2, 2)
    frac = np.mean([se.mahalanobis_squared(x, mu, Sigma) <= k**2 for x in X])
    plotting.plot_ellipse(ax, mu, Sigma, p, label=f"{k}σ  theory {p:.3f}  sampled {frac:.3f}")
ax.legend(fontsize=8); plt.show()

est = se.sample_mean_cov(X)
print("sample mean", est.mean.round(3), "\nsample covariance\n", est.cov.round(3))

### ✏️ Exercise 1 — Mahalanobis distance without an inverse

Implement `mahalanobis_np(x, mu, Sigma)` in NumPy **without** `np.linalg.inv`: factor $\Sigma = LL^T$
(`np.linalg.cholesky`), solve $Ly = x-\mu$ (`np.linalg.solve` or `scipy.linalg.solve_triangular`), return $y^Ty$.

In [ ]:
def mahalanobis_np(x, mu, Sigma):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
for x in X[:20]:
    assert np.isclose(mahalanobis_np(x, mu, Sigma), se.mahalanobis_squared(x, mu, Sigma))
print("✔ matches the C++ implementation")

## Linear propagation is exact

Push the samples through $y = Ax + b$ and compare the sample covariance with $A\Sigma A^T$ (eq. 1.7).

In [ ]:
A = np.array([[np.cos(0.8), -np.sin(0.8)], [np.sin(0.8), np.cos(0.8)]]) @ np.diag([2.0, 0.5])
b = np.array([3.0, -1.0])
Y = X @ A.T + b
exact = se.linear_transform(se.Gaussian(mu, Sigma), A, b)
emp = se.sample_mean_cov(Y)
print("A Σ Aᵀ =\n", exact.cov.round(3), "\nsampled =\n", emp.cov.round(3))

ax = plotting.new_axes()
ax.plot(Y[:, 0], Y[:, 1], ".", ms=2, color="0.6")
plotting.plot_ellipse(ax, exact.mean, exact.cov, 0.95, label="A Σ Aᵀ (95 %)")
plotting.plot_ellipse(ax, emp.mean, emp.cov, 0.95, ls="--", label="sample (95 %)")
ax.legend(); plt.show()

### ✏️ Exercise 2 — Bayes' rule, repeated

The door example of §1.5: prior $p(\text{open}) = 0.5$, $p(z{=}\text{open}\mid\text{open}) = 0.6$,
$p(z{=}\text{open}\mid\text{closed}) = 0.3$. Write `belief_after(n)` that applies `se.bayes_rule` $n$ times
(one "open" reading each time) and returns $p(\text{open})$. Then derive the closed form on paper:
after $n$ readings the odds are multiplied by $(0.6/0.3)^n$, so $p = 1/(1 + 2^{-n})$.

In [ ]:
def belief_after(n):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
assert np.isclose(belief_after(1), 2 / 3)
for n in range(8):
    assert np.isclose(belief_after(n), 1 / (1 + 2.0**-n))
print([round(float(belief_after(n)), 4) for n in range(6)])

### ✏️ Exercise 3 — fusing two scalar estimates

Two independent range estimates of the same wall: $\mathcal N(10, 4)$ (odometry) and $\mathcal N(12, 1)$
(sonar). Use the information form (eq. 1.18–1.19): $1/\sigma^2 = 1/\sigma_a^2 + 1/\sigma_b^2$ and
$\mu/\sigma^2 = \mu_a/\sigma_a^2 + \mu_b/\sigma_b^2$. Return `(mean, var)`.

In [ ]:
def fuse_scalar(ma, va, mb, vb):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
m, v = fuse_scalar(10, 4, 12, 1)
c = se.fuse(se.Gaussian(np.array([10.0]), np.array([[4.0]])), se.Gaussian(np.array([12.0]), np.array([[1.0]])))
assert np.isclose(m, 11.6) and np.isclose(v, 0.8)
assert np.isclose(m, c.mean[0]) and np.isclose(v, c.cov[0, 0])
print(f"fused: N({m:.2f}, {v:.2f}) — more certain than either input")

## 🔨 Break it — linearising a strong nonlinearity

A range–bearing reading at $\rho = 1$ m with bearing noise $\sigma_\phi$ is converted to Cartesian
coordinates. Linearisation (eq. 1.9) keeps the mean at $(1, 0)$; the true mean is $(e^{-\sigma_\phi^2/2}, 0)$.
Increase $\sigma_\phi$ and watch the linearised ellipse leave the samples.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
f = lambda p: se.polar_to_cartesian(p)
Jf = lambda p: se.polar_to_cartesian_jacobian(p)
for ax, s_phi in zip(axes, (0.05, 0.2, 0.5, 1.0)):
    polar = se.Gaussian(np.array([1.0, 0.0]), np.diag([0.02**2, s_phi**2]))
    P = se.sample_gaussian(polar.mean, polar.cov, 4000, se.Rng(1))
    XY = np.array([f(p) for p in P])
    lin, mc = se.propagate_linearized(f, polar, Jf), se.sample_mean_cov(XY)
    plotting.new_axes(ax)
    ax.plot(XY[:, 0], XY[:, 1], ".", ms=1.5, color="0.6")
    plotting.plot_ellipse(ax, lin.mean, lin.cov, 0.95, color="C3")
    plotting.plot_ellipse(ax, mc.mean, mc.cov, 0.95, color="C0")
    ax.set_title(f"σφ = {s_phi}: mean x {mc.mean[0]:.3f}\n(closed form {np.exp(-s_phi**2 / 2):.3f}, linearised 1)",
                 fontsize=9)
plt.tight_layout(); plt.show()

## 🔨 Break it — a gate that ignores correlation

Gate samples of $\mathcal N(\mu,\Sigma)$ at the 95 % level, once with the true $\Sigma$ and once with only its
diagonal (a common shortcut). With the true covariance about 5 % of genuine samples are rejected; with the
diagonal the rejection rate is wrong, so the gate both rejects good data and accepts outliers.

In [ ]:
gate = se.chi2_quantile(0.95, 2)
for name, S in (("true Σ", Sigma), ("diagonal only", np.diag(np.diag(Sigma)))):
    rejected = np.mean([se.mahalanobis_squared(x, mu, S) > gate for x in X])
    print(f"{name:14s}: rejected {100 * rejected:5.1f} % of genuine samples (expected 5.0 %)")

## What to remember

- Covariances add for independent errors; a linear map transforms them as $A\Sigma A^T$, exactly.
- Linearisation $J\Sigma J^T$ is an approximation that fails when the function bends within the spread of $x$.
- Bayes' rule multiplies prior and likelihood and renormalises; two Gaussians fuse into a sharper Gaussian.
- $d_M^2 \sim \chi^2_n$: gates and consistency tests come from its quantiles; a 2-D 1σ ellipse holds only 39 %.